In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
"""
================================================================================
FINAL YEAR PROJECT (FYP) SIMULATION FRAMEWORK
Title: IOT BASED PREDICTIVE MAINTENANCE OF MACHINES POWERED WITH AI AND ML ALGORITHMS
Author: Vincent Tay Yong Jun
Date: Jan 2026
Description:
  - Focused on IMS Dataset (Run-to-Fail).
  - Implements 3-way split (Train/Val/Test).
  - Implements Model-Specific Data Routing (Supervised vs Unsupervised).
  - Runs Hardware Simulation.
  - SAVES results to .csv files.
================================================================================
"""

import os
import time
import numpy as np
import pandas as pd
import scipy.io
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

# =============================================================================
# [USER CONFIGURATION] PATHS
# =============================================================================
DATASET_PATHS = {
"IMS": r"/content/drive/MyDrive/Colab Notebooks/FYP/Datasets/IMS/2nd_test"
}

# CONFIGURATION
WINDOW_SIZE = 1024
STRIDE = 1024
MAX_SAMPLES_PER_CLASS = 1000

# =============================================================================
# 1. SIGNAL PROCESSING
# =============================================================================
def create_sliding_windows(signal, window_size, stride):
    """Creates overlapping windows from the raw time-series signal."""
    if len(signal) < window_size:
        return [np.pad(signal, (0, window_size - len(signal)), 'constant')]
    num_windows = (len(signal) - window_size) // stride + 1
    windows = []
    for i in range(num_windows):
        start = i * stride
        end = start + window_size
        windows.append(signal[start:end])
    return windows

def apply_fft(window):
    """Applies Fast Fourier Transform to extract frequency domain features."""
    fft_values = np.abs(np.fft.rfft(window))
    return fft_values[1:513].reshape(512, 1)

def inject_noise(signal, noise_level):
    """Injects Gaussian noise for robustness testing."""
    if noise_level <= 0: return signal
    noise = np.random.normal(0, 1, signal.shape)
    signal_amplitude = np.max(np.abs(signal))
    return signal + (noise * noise_level * signal_amplitude)

# =============================================================================
# 2. DATA LOADER
# =============================================================================
class UniversalDataLoader:
    def __init__(self, name):
        self.name = name
        self.path = DATASET_PATHS.get(name, "")

    def load_data(self, noise_level=0.0):
        print(f"   [{self.name}] Loading Data (Noise {int(noise_level*100)}%)...")
        if not os.path.exists(self.path):
            print(f"Path not found: {self.path}")
            return None, None

        X_data, y_data = [], []
        counts = {0: 0, 1: 0} # Using dictionary for O(1) tracking

        def process_signal(raw_sig, label):
            if counts[label] >= MAX_SAMPLES_PER_CLASS: return # Exit early if class is full

            noisy_sig = inject_noise(raw_sig, noise_level)
            windows = create_sliding_windows(noisy_sig, WINDOW_SIZE, STRIDE)
            for w in windows:
                if counts[label] >= MAX_SAMPLES_PER_CLASS: break
                X_data.append(apply_fft(w))
                y_data.append(label)
                counts[label] += 1

        try:
            if self.name == "IMS":
                files = sorted(os.listdir(self.path))

                # Based on academic consensus, File 533 marks the onset of early bearing degradation.
                # Index 532 corresponds to the 533rd file (since Python is 0-indexed).
                DEGRADATION_POINT = 532

                # Sample every 5th file to ensure full lifecycle coverage without memory overflow
                sampled_files = files[::5]

                for f in sampled_files:
                    # Recalculate original index to assign the correct label
                    original_index = files.index(f)

                    # Labeling logic: Files prior to degradation point = Healthy (0), otherwise = Anomaly (1)
                    label = 0 if original_index < DEGRADATION_POINT else 1

                    try:
                        # Load the 1-second raw vibration sample
                        df = pd.read_csv(os.path.join(self.path, f), sep='\t', header=None)

                        # CRITICAL: Extract only Column 0 (Bearing 1) where the outer race defect occurred
                        process_signal(df.iloc[:, 0].values, label)

                        # Terminate file reading early if both classes have reached the required sample size
                        if counts[0] >= MAX_SAMPLES_PER_CLASS and counts[1] >= MAX_SAMPLES_PER_CLASS:
                            break
                    except Exception as e:
                        print(f"Error processing {f}: {e}")

            return np.array(X_data), np.array(y_data)

        except Exception as e:
            print(f"Error loading {self.name}: {e}")
            return None, None

# =============================================================================
# 3. MODELS & HARDWARE PROFILES
# =============================================================================
def build_and_train_model(m_name, X_train, y_train, X_val, y_val, num_classes=2, class_weights=None):
    """
    Builds and fits the model based on the specific algorithm type.
    Handles both Supervised and Unsupervised (AE) training logic.
    """
    X_train_flat = X_train.reshape(X_train.shape[0], -1)

    # --- SUPERVISED ML MODELS ---
    if m_name == "Random Forest":
        # Added class_weight='balanced' to handle any residual imbalance
        m = RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced')
        m.fit(X_train_flat, y_train)
        return m

    elif m_name == "SVM":
        # Added class_weight='balanced'
        m = SVC(kernel='rbf', class_weight='balanced')
        m.fit(X_train_flat, y_train)
        return m

    # --- DEEP LEARNING ARCHITECTURES ---
    inp = layers.Input(shape=(512, 1))

    if m_name == "1D-CNN":
        x = layers.Conv1D(32, 16, activation='relu', padding='same')(inp)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(64, 8, activation='relu', padding='same')(x)
        x = layers.GlobalMaxPooling1D()(x)
        out = layers.Dense(num_classes, activation='softmax')(x)
        loss_fn = 'sparse_categorical_crossentropy'

    elif m_name == "MobileNetV2 (1D)":
        x = layers.Conv1D(16, 3, strides=2, padding='same', activation='relu')(inp)
        x = layers.SeparableConv1D(32, 3, padding='same', activation='relu')(x)
        x = layers.GlobalAveragePooling1D()(x)
        out = layers.Dense(num_classes, activation='softmax')(x)
        loss_fn = 'sparse_categorical_crossentropy'

    elif m_name == "LSTM-AE":
        x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp)
        x = layers.LSTM(32, return_sequences=True)(x)
        x = layers.GlobalMaxPooling1D()(x)
        out = layers.Dense(512, activation='linear')(x) # Output must match input shape for AE
        out = layers.Reshape((512, 1))(out)
        loss_fn = 'mse' # Autoencoder uses Mean Squared Error

    elif m_name == "Lite-Hybrid-AE":
        x = layers.Conv1D(filters=16, kernel_size=3, activation='relu', padding='same')(inp)
        x = layers.MaxPooling1D(pool_size=4)(x)
        x = layers.LSTM(32, return_sequences=True, unroll=True)(x)
        x = layers.GlobalMaxPooling1D()(x)
        out = layers.Dense(512, activation='linear')(x)
        out = layers.Reshape((512, 1))(out)
        loss_fn = 'mse'

    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    # --- MODEL SPECIFIC TRAINING LOGIC ---
    if m_name in ["LSTM-AE", "Lite-Hybrid-AE"]:
        # Autoencoder strictly learns to reconstruct its own input
        model.fit(X_train, X_train, validation_data=(X_val, X_val),
                  epochs=50, batch_size=32, verbose=0, callbacks=[early_stop])
    else:
        # Supervised models use class weights to prevent Mode Collapse (F1=0.0)
        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=50, batch_size=32, class_weight=class_weights,
                  verbose=0, callbacks=[early_stop])

    return model

# DEVICE PROFILES
DEVICE_PROFILES = {
    "Raspberry Pi 4": {"speed": 0.25, "tpu": False},
    "Raspberry Pi 5": {"speed": 0.65, "tpu": False},
    "Jetson Nano":    {"speed": 0.45, "tpu": True},
    "Coral Dev":      {"speed": 0.15, "tpu": True},
    "ESP32-S3":       {"speed": 0.015, "tpu": False}
}

# =============================================================================
# 4. MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    hw_results, noise_results = [], []
    models_list = ["Random Forest", "SVM", "1D-CNN", "LSTM-AE", "MobileNetV2 (1D)", "Lite-Hybrid-AE"]

    datasets_to_run = ["IMS"]

    print("\n=== PHASE 1: HARDWARE SIMULATION (3-WAY SPLIT) ===")
    for d in datasets_to_run:
        loader = UniversalDataLoader(d)
        X_all, y_all = loader.load_data(0.0)

        if X_all is None or len(X_all) == 0:
            print(f"Skipping {d} due to missing data.")
            continue

        # -------------------------------------------------------------
        # DATA SPLIT: 70% Train | 15% Val | 15% Test
        # Using Stratify for Run-to-Fail so all sets see Anomalies
        # -------------------------------------------------------------
        X_temp, X_test, y_temp, y_test = train_test_split(
            X_all, y_all, test_size=0.15, stratify=y_all, random_state=42
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42
        )

        # Calculate universal class weights for supervised models
        classes = np.unique(y_train)
        weights = compute_class_weight('balanced', classes=classes, y=y_train)
        class_weight_dict = dict(zip(classes, weights))

        for m in models_list:
            print(f"--> {d}: {m} Training...")

            # =========================================================
            # ALGORITHM-SPECIFIC DATA ROUTING (IF-ELSE LOOP)
            # =========================================================
            if m in ["LSTM-AE", "Lite-Hybrid-AE"]:
                # FILTER: Unsupervised model only gets Normal data (Label == 0)
                idx_train_normal = np.where(y_train == 0)[0]
                idx_val_normal = np.where(y_val == 0)[0]

                X_train_specific = X_train[idx_train_normal]
                y_train_specific = None # AE doesn't need labels
                X_val_specific = X_val[idx_val_normal]
                y_val_specific = None
                weights_to_pass = None
            else:
                # Supervised models get the mixed data with weights
                X_train_specific = X_train
                y_train_specific = y_train
                X_val_specific = X_val
                y_val_specific = y_val
                weights_to_pass = class_weight_dict

            # Build and Train
            model = build_and_train_model(
                m, X_train_specific, y_train_specific, X_val_specific, y_val_specific,
                num_classes=2, class_weights=weights_to_pass
            )

            # =========================================================
            # PREDICTION & LATENCY CALCULATION ON TEST SET
            # =========================================================
            t0 = time.time()
            X_test_flat = X_test.reshape(X_test.shape[0], -1)

            if m in ["Random Forest", "SVM"]:
                y_pred = model.predict(X_test_flat)
            elif m in ["LSTM-AE", "Lite-Hybrid-AE"]:
                # Calculate dynamic threshold using Validation Set
                val_reconstructions = model.predict(X_val_specific, verbose=0)
                val_mse = np.mean(np.power(X_val_specific - val_reconstructions, 2), axis=(1, 2))
                threshold = np.mean(val_mse) + 3 * np.std(val_mse)

                # Predict on Test Set
                test_reconstructions = model.predict(X_test, verbose=0)
                test_mse = np.mean(np.power(X_test - test_reconstructions, 2), axis=(1, 2))
                y_pred = (test_mse > threshold).astype(int)
            else:
                y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

            total_time_ms = (time.time() - t0) * 1000
            host_latency_per_sample = total_time_ms / len(X_test)
            f1 = f1_score(y_test, y_pred, average='macro')

            print(f"    F1-Score: {f1:.4f} | Base Latency: {host_latency_per_sample:.4f}ms")

            # Store results across hardware profiles
            for dev, prof in DEVICE_PROFILES.items():
                accel = 3.5 if (prof['tpu'] and m not in ["Random Forest", "SVM"]) else 1.0
                final_latency = (host_latency_per_sample / prof['speed']) / accel

                base_load = (50 / prof['speed']) * 0.4
                if prof['tpu'] and m not in ["Random Forest", "SVM"]: base_load *= 0.3
                cpu_load = min(base_load + np.random.uniform(0, 5), 100)

                hw_results.append({
                    "Dataset": d, "Model": m, "Device": dev,
                    "F1-Score": round(f1, 4), "Latency(ms)": round(final_latency, 4),
                    "CPU_Load(%)": round(cpu_load, 1)
                })

    # SAVE RESULTS
    pd.DataFrame(hw_results).to_csv("Train_IMS_Results.csv", index=False)
    
    print("\n✅ SIMULATION COMPLETE. CSV files generated.")